# 第4部分：LightGBM模型训练

**目的：** 用通过IV筛选和单调性检验的特征，训练一个LGBM分类模型

## 为什么选LGBM？

| 对比 | LGBM | XGBoost | 逻辑回归 |
|------|------|---------|---------|
| 速度 | 最快 | 较快 | 快 |
| 精度 | 高 | 高 | 中 |
| 特征交互 | 自动学习 | 自动学习 | 需手动 |
| 可解释性 | 中 | 中 | 高 |
| 业界常用度 | 极高 | 高 | 高 |

In [ ]:
import lightgbm as lgb
from sklearn.model_selection import train_test_split
import numpy as np
import pandas as pd

## 步骤1：准备训练数据

In [ ]:
# 特征矩阵 X 和标签 y
train_data_x = df[monotonic_features]  # 通过筛选的特征
train_data_y = df['dob4_ever10_flg']   # 标签：0=好人, 1=坏人

# 划分训练集和测试集（7:3）
X_train, X_test, y_train, y_test = train_test_split(
    train_data_x, train_data_y,
    test_size=0.3,       # 30%做测试
    random_state=42,     # 固定随机种子，确保可复现
    stratify=train_data_y  # 分层抽样，保持好坏比例一致
)

print(f'训练集: {X_train.shape[0]}样本, 坏账率{y_train.mean():.2%}')
print(f'测试集: {X_test.shape[0]}样本, 坏账率{y_test.mean():.2%}')

## 步骤2：设置LGBM参数

每个参数的含义和调参建议：

In [ ]:
# LGBM参数详解
params_stage1 = {
    'objective': 'binary',      # 二分类任务
    'metric': 'auc',            # 评估指标：AUC（越接近1越好）
    'boosting_type': 'gbdt',    # 第一阶段用GBDT（梯度提升树）

    # ===== 树的复杂度 =====
    'num_leaves': 31,           # 每棵树最多31个叶子（控制复杂度）
                                # 越大->模型越复杂->越容易过拟合
    'max_depth': 5,             # 树的最大深度（5层）
                                # 限制深度防止过拟合

    # ===== 学习速度 =====
    'learning_rate': 0.05,      # 学习率（每棵树的贡献权重）
                                # 越小->需要更多树->但更稳定
    'n_estimators': 200,        # 总共训练200棵树

    # ===== 防过拟合 =====
    'min_child_samples': 50,    # 叶子节点最少50个样本
                                # 防止树学到噪声
    'reg_alpha': 0.1,           # L1正则化（让不重要的特征权重=0）
    'reg_lambda': 1.0,          # L2正则化（让所有特征权重更小）
    'subsample': 0.8,           # 每棵树只用80%的样本
    'colsample_bytree': 0.8,   # 每棵树只用80%的特征

    # ===== 其他 =====
    'is_unbalance': True,       # 自动处理正负样本不平衡
    'random_state': 42,
    'verbose': -1               # 不打印训练日志
}

print("Stage 1参数设置完成")
print(f"预计训练: {params_stage1['n_estimators']}棵树")

## 步骤3：两阶段训练

### 为什么分两阶段？

```
Stage 1 (GBDT): 快速学到主要模式
    |
    v
Stage 2 (DART): 精细调优，防止过拟合

DART = Dropouts meet Multiple Additive Regression Trees
    每次训练时随机"丢弃"一些已有的树
    类似神经网络的Dropout，增强泛化能力
```

In [ ]:
# ===== Stage 1: GBDT快速训练 =====
lgbmodel_s1 = lgb.LGBMClassifier(**params_stage1)
lgbmodel_s1.fit(X_train, y_train)

# 评估Stage 1
from sklearn.metrics import roc_auc_score
auc_train = roc_auc_score(y_train, lgbmodel_s1.predict_proba(X_train)[:, 1])
auc_test = roc_auc_score(y_test, lgbmodel_s1.predict_proba(X_test)[:, 1])
print(f"Stage 1 - Train AUC: {auc_train:.4f}, Test AUC: {auc_test:.4f}")

# ===== Stage 2: DART精细调优 =====
params_stage2 = params_stage1.copy()
params_stage2['boosting_type'] = 'dart'    # 切换到DART
params_stage2['n_estimators'] = 100        # 少一些树
params_stage2['learning_rate'] = 0.02      # 更小的学习率
params_stage2['drop_rate'] = 0.1           # 每轮丢弃10%的树

lgbmodel = lgb.LGBMClassifier(**params_stage2)
lgbmodel.fit(X_train, y_train)

# 评估Stage 2
auc_train2 = roc_auc_score(y_train, lgbmodel.predict_proba(X_train)[:, 1])
auc_test2 = roc_auc_score(y_test, lgbmodel.predict_proba(X_test)[:, 1])
print(f"Stage 2 - Train AUC: {auc_train2:.4f}, Test AUC: {auc_test2:.4f}")

# 过拟合判断：Train和Test的AUC差距
gap = auc_train2 - auc_test2
print(f"\n过拟合程度: {gap:.4f} ({'正常' if gap < 0.03 else '有过拟合风险'})")

---
### 面试考点

| 问题 | 答案 |
|------|------|
| GBDT和DART的区别？ | DART每轮随机丢弃树，类似Dropout，泛化更好 |
| learning_rate为什么设小？ | 小lr + 多树 = 更稳定的收敛，不容易过拟合 |
| is_unbalance做了什么？ | 自动给少数类（坏人）更高权重，不被好人淹没 |
| 怎么判断过拟合？ | Train AUC和Test AUC差距>0.03就有风险 |
| subsample=0.8什么意思？ | 每棵树随机抽80%数据训练，增加多样性 |
| 为什么分两阶段？ | Stage1快速逼近，Stage2精细打磨+防过拟合 |